# Scaling Ambiguity Augmenting Human Annotation in Speech Emotion Recognition with Audio-Language Models - Data Processing

This notebook will process the **IEMOCAP** and **MSP-Podcast datasets** and prepare them for subsequent processing. We will first process the original human annotations to obtain the **Empirical Distribution**.

## Overview
- Load both datasets
- Split the datasets into training and test sets
- Prepare data for fine-tuning

## Environment Setup and Dependency Import

In [ ]:
import sys
sys.path.append('..')

import os
import json
from collections import Counter
from sklearn.model_selection import train_test_split

from lib import load_data as ld

# Section 1: Data Loading

In [2]:
iemocap_data = ld.load_raw_dataset('iemocap')
msp_data = ld.load_raw_dataset('msp')

Loaded 10043 samples from IEMOCAP dataset
Loaded 104266 samples from MSP dataset


# Section 2: Dataset Split (Training and Test Sets)

In [4]:
def filter_and_split_data(data, test_size=0.2, random_state=42):
    """
    Filter data for items with need_prediction='yes' and then split into train/test sets
    """
    # Filter for only need_prediction='yes' samples
    filtered_data = [item for item in data if item['need_prediction'] == 'yes']
    
    # Split the filtered data into training and test sets using sklearn
    train_indices, test_indices = train_test_split(
        range(len(filtered_data)), 
        test_size=test_size, 
        random_state=random_state
    )
    
    train_data = [filtered_data[i] for i in train_indices]
    test_data = [filtered_data[i] for i in test_indices]
    
    return train_data, test_data

# Split IEMOCAP dataset
iemocap_train, iemocap_test = filter_and_split_data(iemocap_data)
print(f"\nIEMOCAP dataset split:")
print(f"Training set samples: {len(iemocap_train)}")
print(f"Test set samples: {len(iemocap_test)}")

# Split MSP-Podcast dataset
msp_train, msp_test = filter_and_split_data(msp_data)
print(f"\nMSP-Podcast dataset split:")
print(f"Training set samples: {len(msp_train)}")
print(f"Test set samples: {len(msp_test)}")


IEMOCAP dataset split:
Training set samples: 3498
Test set samples: 875

MSP-Podcast dataset split:
Training set samples: 3291
Test set samples: 823


# Section 3: Generate Empirical Distribution Annotations

In [5]:
def create_emotion_distributions(data):
    """
    Convert multi-annotator emotion labels to probability distributions
    Maintaining the original data structure
    """
    processed_data = []
    
    for item in data:
        # Create a deep copy of the original item
        processed_item = item.copy()
        
        # Calculate emotion distribution
        if 'emotion' in item and isinstance(item['emotion'], list):
            emotion_counts = Counter(item['emotion'])
            total_annotations = len(item['emotion'])
            
            # Replace the emotion list with a distribution dictionary
            emotion_distribution = {}
            for emotion, count in emotion_counts.items():
                probability = count / total_annotations
                emotion_distribution[emotion] = probability
            
            # Replace the emotion field with the distribution
            processed_item['emotion'] = emotion_distribution
        
        processed_data.append(processed_item)
    
    # Get unique emotions across all samples (for reference)
    all_emotions = set()
    for item in processed_data:
        if isinstance(item['emotion'], dict):
            all_emotions.update(item['emotion'].keys())
    
    return processed_data, sorted(list(all_emotions))

In [6]:
# Generate emotion distributions for training sets
iemocap_train_dist, iemocap_emotions = create_emotion_distributions(iemocap_train)
msp_train_dist, msp_emotions = create_emotion_distributions(msp_train)

# Generate emotion distributions for test sets
iemocap_test_dist, _ = create_emotion_distributions(iemocap_test)
msp_test_dist, _ = create_emotion_distributions(msp_test)

# Section 4: Save Processed Data

In [ ]:
# Create output directory
output_dir = os.path.join('..', 'processed_data')
os.makedirs(output_dir, exist_ok=True)

# Save processed data for later use
def save_to_json(data, filename):
    file_path = os.path.join(output_dir, filename)
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)
    print(f"Saved {len(data)} items to {filename}")


Total filtered samples (need_prediction='yes'):
IEMOCAP: 4373 samples
MSP-Podcast: 4114 samples


In [ ]:
# Save processed data
save_to_json(iemocap_train, 'human_iemocap_train_raw_annotations.json')
save_to_json(iemocap_test, 'iemocap_test_raw_annotations.json')
save_to_json(msp_train, 'human_msp_train_raw_annotations.json')
save_to_json(msp_test, 'msp_test_raw_annotations.json')

save_to_json(iemocap_train_dist, 'human_iemocap_train_distributions.json')
save_to_json(iemocap_test_dist, 'iemocap_test_distributions.json')
save_to_json(msp_train_dist, 'human_msp_train_distributions.json')
save_to_json(msp_test_dist, 'msp_test_distributions.json')

# Save emotion category lists
save_to_json(list(iemocap_emotions), 'iemocap_emotion_classes.json')
save_to_json(list(msp_emotions), 'msp_emotion_classes.json')

Saved 3498 items to human_iemocap_train_distributions.json
Saved 875 items to iemocap_test_distributions.json
Saved 3291 items to human_msp_train_distributions.json
Saved 823 items to msp_test_distributions.json
Saved 4 items to iemocap_emotion_classes.json
Saved 4 items to msp_emotion_classes.json

Data processing complete, all files saved to 'processed_data' directory
